In [21]:
import sqlite3
import pandas as pd

# Input prediction csv and database file and table_name
csv_file = "C:/Users/twans/Documents/GitHub/NBA/Historical Odds/Model Evaluation/cnn_player_predictions.csv"  
database_file = "C:/Users/twans/Documents/GitHub/NBA/Tao Data Collection/1-13_nba_data.db.db"
table_name = "1-13point_predictions"  

# Function to load CSV and insert into the database
def csv_to_db(csv_file_path, db_path, table_name):
    try:
        # Connect to the SQLite database
        conn = sqlite3.connect(database_file)
        cursor = conn.cursor()
        
        # Load the CSV file into a Pandas DataFrame
        df = pd.read_csv(csv_file_path)
        
        # Escape table name for SQL compatibility
        table_name_escaped = f'"{table_name}"'
        
        # Delete table if it already exists
        cursor.execute(f"DROP TABLE IF EXISTS {table_name_escaped}")
        conn.commit()

        # Create table if it doesn't exist
        columns = ', '.join([f'"{col}" TEXT' for col in df.columns])  # Escape column names
        create_table_query = f"CREATE TABLE IF NOT EXISTS {table_name_escaped} ({columns})"
        cursor.execute(create_table_query)

        # Insert data into the table
        df.to_sql(table_name, conn, if_exists='append', index=False)

        # Commit the transaction and close the connection
        conn.commit()
        conn.close()
        print(f"Data successfully added to the table '{table_name}' in the database '{db_path}'")
    except Exception as e:
        print(f"An error occurred: {e}")

csv_to_db(csv_file, database_file, table_name)


Data successfully added to the table '1-13point_predictions' in the database 'C:/Users/twans/Documents/GitHub/NBA/Tao Data Collection/1-13_nba_data.db.db'


In [22]:
# Connect to the SQLite database
conn = sqlite3.connect(database_file)
cursor = conn.cursor()

#Delete fanduel_model_evaluation_2024 if exists already
cursor.execute("DROP TABLE IF EXISTS fanduel_model_evaluation_2024")
conn.commit()

# Properly escape table names and handle leading zeroes using ltrim
combine_query = """
CREATE TABLE fanduel_model_evaluation_2024 AS
SELECT pp.*, pap.*
FROM "1-13point_predictions" AS pp
INNER JOIN "combined_fanduel_prop_and_players" AS pap
ON ltrim(pp."GAME_ID", '0') = ltrim(pap."GAME_ID", '0') 
   AND pp."PLAYER_ID" = pap."PLAYER_ID";
"""

# Execute the query
cursor.execute(combine_query)

# Commit the transaction and close the connection
conn.commit()
conn.close()

print('Combined predictions with lines successfully, accounting for string-based GAME_ID.')


Combined predictions with lines successfully, accounting for string-based GAME_ID.


In [23]:
# Connect to the SQLite database
conn = sqlite3.connect(database_file)
cursor = conn.cursor()

# Add the new column (if it doesn't already exist)
try:
    cursor.execute("ALTER TABLE fanduel_model_evaluation_2024 ADD COLUMN line_result INTEGER;")
except sqlite3.OperationalError:
    print("Column 'line_result' already exists.")

# Update the table with the logic
update_query = """
UPDATE fanduel_model_evaluation_2024
SET line_result = 
    CASE
        WHEN outcome_name = 'Over' AND CAST(PTS AS REAL) > point THEN 1
        WHEN outcome_name = 'Under' AND CAST(PTS AS REAL) < point THEN 1
        ELSE 0
    END;
"""
cursor.execute(update_query)

# Commit the transaction and close the connection
conn.commit()
conn.close()

print("Updated 'actual_result' column successfully.")

Updated 'actual_result' column successfully.


In [24]:
# Connect to the SQLite database
conn = sqlite3.connect(database_file)
cursor = conn.cursor()

# Add the new column (if it doesn't already exist)
cursor.execute("ALTER TABLE fanduel_model_evaluation_2024 ADD COLUMN pred_line_result INTEGER;")

# Update the table with the logic
update_query = """
UPDATE fanduel_model_evaluation_2024
SET pred_line_result = 
    CASE
        WHEN outcome_name = 'Over' AND CAST(Predicted_PTS AS REAL) > point THEN 1
        WHEN outcome_name = 'Under' AND CAST(Predicted_PTS AS REAL) < point THEN 1
        ELSE 0
    END;
"""
cursor.execute(update_query)

# Commit the transaction and close the connection
conn.commit()
conn.close()

print("Updated 'actual_result' column successfully.")

Updated 'actual_result' column successfully.


In [26]:
#SQL Query to find accuracy

#SELECT COUNT(*) AS matching_rows
# FROM fanduel_model_evaluation_2024
# WHERE line_result = pred_line_result;